# <span style="color:orange"> **NOTES** </span>
## <span style="color:yellow"> ***WEEK 2*** </span>
### <span style="color:red"> **Building Makemore Part 2** </span>
1. We try to make a multi-layer perceptron, which takes a context of N characters and tries to predict the next character. We take each character and encode them in a d-dimensional space. The characters that are used inter-changeably are likely to lie closer in this space. We have an look-up table that encodes the context into this space. Then this is fed into a hidden layer whose number of neurons is a hyper-parameter. This is further connected into a output layer through tanh non-linearity. This output layer outputs a logit which is soft-maxxed to get the probability distribution.
2. The parameters are the weights and biases of the output layer, the hidden layer and the look-up table.

In [60]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline
import random
random.seed(4267)

In [61]:
words=open('names.txt','r').read().splitlines()

In [62]:
#hyperparameters
block_size=4
vector_dim=10
hidden_layer_size=200
mini_batch_size=32

In [63]:
#encoding
itos={i+1:ch for i,ch in enumerate(list("abcdefghijklmnopqrstuvwxyz"))}
itos[0]='.'
stoi={ch:i for i,ch in itos.items()}

We divide the words into 3 splits:
1. Training Split: for training parameters (80%)
2. Dev Split: for training hyperparameters (10%)
3. Test Split: for evaluating model (10%)

In [71]:
#building the dataset
random.shuffle(words)
n1=int(0.8*len(words))
n2=int(0.9*len(words))
train_words=words[:n1]
dev_words=words[n1:n2]
test_words=words[n2:]
Xtr=[]
Ytr=[]
Xdev=[]
Ydev=[]
Xte=[]
Yte=[]
context=[0]*block_size
for word in train_words:
    for ch in word+".":
        Xtr.append(context)
        Ytr.append(stoi[ch])
        context=context[1:]+[stoi[ch]]
for word in dev_words:
    for ch in word+".":
        Xdev.append(context)
        Ydev.append(stoi[ch])
        context=context[1:]+[stoi[ch]]
for word in test_words:
    for ch in word+".":
        Xte.append(context)
        Yte.append(stoi[ch])
        context=context[1:]+[stoi[ch]]

Xtr=torch.tensor(Xtr)
Ytr=torch.tensor(Ytr)

Xte=torch.tensor(Xte)
Yte=torch.tensor(Yte)

Xdev=torch.tensor(Xdev)
Ydev=torch.tensor(Ydev)


In [72]:
g=torch.Generator().manual_seed(2176)

In [73]:
#the look-up table
C=torch.randn((27,vector_dim),generator=g)

In [74]:
#the hidden layer
W1=torch.randn((vector_dim*block_size,hidden_layer_size),generator=g)
b1=torch.randn(hidden_layer_size,generator=g)

In [75]:
#final layer
W2=torch.randn((hidden_layer_size,27),generator=g)*0.1
b2=torch.randn(27,generator=g)*0

In [76]:
parameters=[C,W1,b1,W2,b2]
for p in parameters:
    p.requires_grad=True
print("Total Parameters:",sum(p.nelement() for p in parameters))

Total Parameters: 13897


One way to embedd an integer into the vector space is one hot encoding. The other way is to index. Both methods are same. Also Pytorch indexing works with a tensor.

1. We use the torch.arange() to generate the row indices for the corresponding column indices in Y. In torch the tensor indexing works as follows [row indices,colun indices].

2. Instead of the loss calculation from logits explicitly in multi steps there is a pytorch function torch.nn.functional.cross_entropy(logits, Y) which returns the same value. This is very effecient in backward pass. This is also numerically well behaved. Because when we exp() the logits sometimes it maybe that we get nan. So we need to subtract the max logit value from each field of the logit to make it well behaved.

3. We have to lot of work to go through the whole batch during each training iteration. Instead we divide our data into mini batches randomly. It is seen that it is quite efficient.

4. To figure out the learning rate, first we see some extreme values like 10 or 0.00001 and find out for what values is the loss getting resonable. Then we plot for learning rates in some range and look at the minima

In [ ]:
#training loop
lre=torch.linspace(-3,0,1000)
lossi=[]
lrei=[]
for i in range(250000):
    #lr=10**lre[i]
    lr=0.1 if i <150000 else 0.01
    #minibatch
    ix=torch.randint(0,len(Ytr),(mini_batch_size,))
    #forward pass
    logits=torch.tanh(C[Xtr[ix]].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    loss=torch.nn.functional.cross_entropy(logits,Ytr[ix])
    #backward pass
    for p in parameters:
        p.grad=None
    loss.backward()
    #update
    for p in parameters:
        p.data-=lr*p.grad
#overall loss
logits=torch.tanh(C[X].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
torch.nn.functional.cross_entropy(logits,Y)



IndexError: index 1000 is out of bounds for dimension 0 with size 1000

In [78]:
#training loss
logits=torch.tanh(C[Xtr].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
torch.nn.functional.cross_entropy(logits,Ytr)


tensor(2.0585, grad_fn=<NllLossBackward0>)

In [79]:
#dev loss
logits=torch.tanh(C[Xdev].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
torch.nn.functional.cross_entropy(logits,Ydev)

tensor(2.1426, grad_fn=<NllLossBackward0>)

In [ ]:
#testing loss
logits=torch.tanh(C[Xte].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
torch.nn.functional.cross_entropy(logits,Yte)

tensor(2.1457, grad_fn=<NllLossBackward0>)

In [36]:
def model(x):
    logit=torch.tanh(C[x].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    prob=torch.softmax(logit,dim=-1)
    return itos[torch.multinomial(prob,1,replacement=True,generator=g).item()]


In [ ]:
#sampling model 1
with torch.inference_mode():
    for _ in range(10):
        word=''
        context=[0]*block_size
        while True:
            ch=model(context)
            word+=ch
            context=context[1:]+[stoi[ch]]
            if ch=='.':
                break
        print(word)


railiyah.
riddickl.
ilai.
vughall.
ixdor.
erson.
dhndie.
aeolanna.
arrien.
yrrishika.


### <span style="color:red"> **Exercises (MLP)** </span>
### <span style="color:cyan"> E01.Tune the hyperparameters to beat Andrej's validation loss of 2.2 </span>


We brought  down the validation loss to **2.14**.

In [ ]:
#hyperparameters
block_size=4
vector_dim=10
hidden_layer_size=200
mini_batch_size=32

In [185]:
#building the dataset
random.shuffle(words)
n1=int(0.8*len(words))
n2=int(0.9*len(words))
train_words=words[:n1]
dev_words=words[n1:n2]
test_words=words[n2:]
Xtr=[]
Ytr=[]
Xdev=[]
Ydev=[]
Xte=[]
Yte=[]

for word in train_words:
    context=[0]*block_size
    for ch in word+".":
        Xtr.append(context)
        Ytr.append(stoi[ch])
        context=context[1:]+[stoi[ch]]
for word in dev_words:
    context=[0]*block_size
    for ch in word+".":
        Xdev.append(context)
        Ydev.append(stoi[ch])
        context=context[1:]+[stoi[ch]]
for word in test_words:
    context=[0]*block_size
    for ch in word+".":
        Xte.append(context)
        Yte.append(stoi[ch])
        context=context[1:]+[stoi[ch]]

Xtr=torch.tensor(Xtr)
Ytr=torch.tensor(Ytr)

Xte=torch.tensor(Xte)
Yte=torch.tensor(Yte)

Xdev=torch.tensor(Xdev)
Ydev=torch.tensor(Ydev)


In [186]:
#the look-up table
C=torch.randn((27,vector_dim),generator=g)
#the hidden layer
W1=torch.randn((vector_dim*block_size,hidden_layer_size),generator=g)
b1=torch.randn(hidden_layer_size,generator=g)
#final layer
W2=torch.randn((hidden_layer_size,27),generator=g)
b2=torch.randn(27,generator=g)

parameters=[C,W1,b1,W2,b2]
for p in parameters:
    p.requires_grad=True
print("Total Parameters:",sum(p.nelement() for p in parameters))


Total Parameters: 13897


In [187]:
#training loop
max_steps=200000
for i in range(max_steps):
    lr=0.1 if i <100000 else 0.01
    #minibatch
    ix=torch.randint(0,len(Ytr),(mini_batch_size,))
    #forward pass
    logits=torch.tanh(C[Xtr[ix]].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    loss=torch.nn.functional.cross_entropy(logits,Ytr[ix])
    #backward pass
    for p in parameters:
        p.grad=None
    loss.backward()
    #update
    for p in parameters:
        p.data-=lr*p.grad
    #stats
    if i % 10000 == 0: # print every once in a while
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  
#overall loss
with torch.no_grad():
    logits=torch.tanh(C[Xtr].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    torch.nn.functional.cross_entropy(logits,Ytr)

      0/ 200000: 22.2332
  10000/ 200000: 2.2706
  20000/ 200000: 2.6287
  30000/ 200000: 2.5053
  40000/ 200000: 2.3547
  50000/ 200000: 2.5850
  60000/ 200000: 2.2809
  70000/ 200000: 2.4934
  80000/ 200000: 2.0861
  90000/ 200000: 1.8812
 100000/ 200000: 2.6446
 110000/ 200000: 1.8106
 120000/ 200000: 2.3915
 130000/ 200000: 2.2670
 140000/ 200000: 1.7965
 150000/ 200000: 2.5904
 160000/ 200000: 2.0955
 170000/ 200000: 2.1515
 180000/ 200000: 2.1451
 190000/ 200000: 2.0975


In [188]:
#training loss
with torch.no_grad():
    logits=torch.tanh(C[Xtr].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    print(torch.nn.functional.cross_entropy(logits,Ytr))



tensor(2.0906)


In [189]:
#dev loss
with torch.no_grad():
    logits=torch.tanh(C[Xdev].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    print(torch.nn.functional.cross_entropy(logits,Ydev))

tensor(2.1448)


In [190]:
#testing loss
with torch.no_grad():
    logits=torch.tanh(C[Xte].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    print(torch.nn.functional.cross_entropy(logits,Yte))

tensor(2.1412)


In [191]:
def model(x):
    logit=torch.tanh(C[x].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    prob=torch.softmax(logit,dim=-1)
    return itos[torch.multinomial(prob,1,replacement=True,generator=g).item()]


In [192]:
#sampling
with torch.inference_mode():
    for _ in range(10):
        word=''
        context=[0]*block_size
        while True:
            ch=model(context)
            word+=ch
            context=context[1:]+[stoi[ch]]
            if ch=='.':
                break
        print(word)


deomora.
carisda.
qwester.
mcon.
ella.
avriana.
amaanell.
kiyonna.
zyan.
brymeei.



### <span style="color:cyan"> E02. Initialization matters. What is the loss you'd get if the predicted probabilities at initialization were perfectly uniform? What loss do we actually achieve? Can u tune the initialization to get a starting loss that is much more similar to the uniform loss?</span><br>
1. The loss at uniform initialisation would be -log(1/27)= ***3.296***<br>
2. Instead our network is confidently wrong and produces a high loss of around ***20-30***.<br>
3. We can tune the initialisation to get an initial loss of around the uniform loss by making the last layers weights and biases approx zero. We multiply the weights by a small number like 0.1. We dont make the weights exactly zero as that would take away the randomness which is helpful.

In [219]:
#the look-up table
C=torch.randn((27,vector_dim),generator=g)
#the hidden layer
W1=torch.randn((vector_dim*block_size,hidden_layer_size),generator=g)
b1=torch.randn(hidden_layer_size,generator=g)
#final layer
W2=torch.randn((hidden_layer_size,27),generator=g)*0.1
b2=torch.randn(27,generator=g)*0

parameters=[C,W1,b1,W2,b2]
for p in parameters:
    p.requires_grad=True
print("Total Parameters:",sum(p.nelement() for p in parameters))


Total Parameters: 15897


In [220]:
#training loop
max_steps=200000
for i in range(max_steps):
    lr=0.1 if i <100000 else 0.01
    #minibatch
    ix=torch.randint(0,len(Ytr),(mini_batch_size,))
    #forward pass
    logits=torch.tanh(C[Xtr[ix]].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    loss=torch.nn.functional.cross_entropy(logits,Ytr[ix])
    #backward pass
    for p in parameters:
        p.grad=None
    loss.backward()
    #update
    for p in parameters:
        p.data-=lr*p.grad
    #stats
    if i % 10000 == 0: # print every once in a while
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  
#overall loss
with torch.no_grad():
    logits=torch.tanh(C[Xtr].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    torch.nn.functional.cross_entropy(logits,Ytr)

      0/ 200000: 3.8685
  10000/ 200000: 1.9052
  20000/ 200000: 2.5268
  30000/ 200000: 2.1142
  40000/ 200000: 2.5351
  50000/ 200000: 2.3038
  60000/ 200000: 1.7530
  70000/ 200000: 2.2373
  80000/ 200000: 2.4379
  90000/ 200000: 2.1583
 100000/ 200000: 2.0505
 110000/ 200000: 1.9049
 120000/ 200000: 1.9773
 130000/ 200000: 2.1221
 140000/ 200000: 1.8665
 150000/ 200000: 2.1047
 160000/ 200000: 2.2748
 170000/ 200000: 2.4236
 180000/ 200000: 2.2621
 190000/ 200000: 1.9261


In [221]:
#training loss
with torch.no_grad():
    logits=torch.tanh(C[Xtr].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    print(torch.nn.functional.cross_entropy(logits,Ytr))



tensor(2.0264)


In [222]:
#dev loss
with torch.no_grad():
    logits=torch.tanh(C[Xdev].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    print(torch.nn.functional.cross_entropy(logits,Ydev))

tensor(2.1034)


The starting loss being smaller helps us to get a better model in the end.

### <span style="color:cyan"> E03. Read the Bengio et al. 2003 paper (link above). Implement and try any idea from the paper. Did it work?</span><br>
1. By increasing the context to 5 we got a loss of ***2.098***


In [246]:
#hyperparameters
block_size=5
vector_dim=10
hidden_layer_size=200
mini_batch_size=32

In [247]:
#building the dataset
random.shuffle(words)
n1=int(0.8*len(words))
n2=int(0.9*len(words))
train_words=words[:n1]
dev_words=words[n1:n2]
test_words=words[n2:]
Xtr=[]
Ytr=[]
Xdev=[]
Ydev=[]
Xte=[]
Yte=[]

for word in train_words:
    context=[0]*block_size
    for ch in word+".":
        Xtr.append(context)
        Ytr.append(stoi[ch])
        context=context[1:]+[stoi[ch]]
for word in dev_words:
    context=[0]*block_size
    for ch in word+".":
        Xdev.append(context)
        Ydev.append(stoi[ch])
        context=context[1:]+[stoi[ch]]
for word in test_words:
    context=[0]*block_size
    for ch in word+".":
        Xte.append(context)
        Yte.append(stoi[ch])
        context=context[1:]+[stoi[ch]]

Xtr=torch.tensor(Xtr)
Ytr=torch.tensor(Ytr)

Xte=torch.tensor(Xte)
Yte=torch.tensor(Yte)

Xdev=torch.tensor(Xdev)
Ydev=torch.tensor(Ydev)


In [248]:
#the look-up table
C=torch.randn((27,vector_dim),generator=g)
#the hidden layer
W1=torch.randn((vector_dim*block_size,hidden_layer_size),generator=g)
b1=torch.randn(hidden_layer_size,generator=g)
#final layer
W2=torch.randn((hidden_layer_size,27),generator=g)*0.1
b2=torch.randn(27,generator=g)*0

parameters=[C,W1,b1,W2,b2]
for p in parameters:
    p.requires_grad=True
print("Total Parameters:",sum(p.nelement() for p in parameters))


Total Parameters: 15897


In [249]:
#training loop
max_steps=250000
for i in range(max_steps):
    lr=0.1 if i <150000 else 0.01
    #minibatch
    ix=torch.randint(0,len(Ytr),(mini_batch_size,))
    #forward pass
    logits=torch.tanh(C[Xtr[ix]].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    loss=torch.nn.functional.cross_entropy(logits,Ytr[ix])
    #backward pass
    for p in parameters:
        p.grad=None
    loss.backward()
    #update
    for p in parameters:
        p.data-=lr*p.grad
    #stats
    with torch.no_grad():

        if i % 10000 == 0: # print every once in a while
            print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
            logits=torch.tanh(C[Xdev].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
            print('validation loss:',torch.nn.functional.cross_entropy(logits,Ydev).item())
  
#overall loss
with torch.no_grad():
    logits=torch.tanh(C[Xtr].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    torch.nn.functional.cross_entropy(logits,Ytr)

      0/ 250000: 4.0212
validation loss: 4.062397003173828
  10000/ 250000: 1.9984
validation loss: 2.3361470699310303
  20000/ 250000: 1.6462
validation loss: 2.3013205528259277
  30000/ 250000: 2.5325
validation loss: 2.2873620986938477
  40000/ 250000: 2.4010
validation loss: 2.2869770526885986
  50000/ 250000: 2.2350
validation loss: 2.2829532623291016
  60000/ 250000: 2.2024
validation loss: 2.273839235305786
  70000/ 250000: 2.5842
validation loss: 2.2570788860321045
  80000/ 250000: 2.6586
validation loss: 2.262556552886963
  90000/ 250000: 1.8308
validation loss: 2.2649009227752686
 100000/ 250000: 2.2590
validation loss: 2.2345969676971436
 110000/ 250000: 1.8634
validation loss: 2.2737674713134766
 120000/ 250000: 2.2369
validation loss: 2.2334530353546143
 130000/ 250000: 2.2085
validation loss: 2.2377257347106934
 140000/ 250000: 2.4524
validation loss: 2.208810329437256
 150000/ 250000: 2.1950
validation loss: 2.2276997566223145
 160000/ 250000: 1.8416
validation loss: 2.1

In [250]:
#training loss
with torch.no_grad():
    logits=torch.tanh(C[Xtr].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    print(torch.nn.functional.cross_entropy(logits,Ytr))



tensor(2.0024)


In [251]:
#dev loss
with torch.no_grad():
    logits=torch.tanh(C[Xdev].view((-1,block_size*vector_dim)) @ W1 + b1) @ W2 + b2
    print(torch.nn.functional.cross_entropy(logits,Ydev))

tensor(2.1184)
